# TriCheck-LK - Multilingual Text Preprocessing

This notebook preprocesses English, Sinhala and Tamil government circular text
before cross-lingual semantic analysis.

The preprocessing pipeline preserves multilingual meaning while removing
extraction-related noise.

Main steps:
- Unicode normalization
- Control-character removal
- Whitespace normalization
- Preservation of punctuation and numbers
- Preparation for semantic chunking

In [8]:
import pandas as pd 
import re
import unicodedata

In [5]:
path="C:/Users/User/Desktop/TriCheck_LK/data/processed/extracted_documents_clean.jsonl"

In [10]:
df=pd.read_json(path,lines=True)
print("Dataset Shape:",df.shape)

Dataset Shape: (4656, 8)


Preprocessing Function

In [11]:
training_doc = df[
    (df["circular_id"] == 2221)
    &
    (df["language"] == "English")
].iloc[0]

In [4]:
def clean_text(text):

    # Convert to string safely
    text = str(text)

    # Normalize Unicode characters
    # Important for Sinhala and Tamil text
    text = unicodedata.normalize(
        "NFC",
        text
    )

    # Remove unwanted control characters
    # Keep newline, tab, ZWNJ and ZWJ because they can be
    # meaningful in Sinhala and other Indic-script text
    text = "".join(
        char
        for char in text
        if (
            unicodedata.category(char)[0] != "C"
            or char in "\n\t\u200c\u200d"
        )
    )

    # Replace tabs with spaces
    text = text.replace(
        "\t",
        " "
    )

    # Remove repeated spaces
    text = re.sub(
        r"[ ]+",
        " ",
        text
    )

    # Remove excessive blank lines
    text = re.sub(
        r"\n\s*\n+",
        "\n",
        text
    )

    # Remove leading/trailing whitespace
    text = text.strip()

    return text

In [12]:
training_clean_text = clean_text(
    training_doc["text"]
)

print(
    "Training raw length:",
    len(training_doc["text"])
)

print(
    "Training cleaned length:",
    len(training_clean_text)
)

Training raw length: 23362
Training cleaned length: 22993


In [14]:
#apply cleaning
df["clean_text"] = (
    df["text"]
    .apply(clean_text)
)

print(
    "Cleaning completed."
)

Cleaning completed.


In [15]:
#comaprison original vs cleaned
sample = df.iloc[0]

print("Language:")
print(sample["language"])

print("\nORIGINAL:")
print("-" * 60)
print(sample["text"][:1000])

print("\nCLEANED:")
print("-" * 60)
print(sample["clean_text"][:1000])

Language:
English

ORIGINAL:
------------------------------------------------------------
Public Administration Circular : 17/2026 
 
 
My number : SLAS/R/Merit/2026 
Ministry of Public Administration, 
Provincial Councils and Local Government  
Independence Square 
Colombo 07. 
 
10.08.2026 
 
Secretaries of Ministries 
Chief Secretaries of Provinces 
Secretaries of Commissions 
District Secretaries / Government Agents 
Heads of Departments. 
 
Competitive Examination for Recruitment to Grade III of the Sri Lanka Administrative Service 
under the Merit Stream -2026 
 
Applications are hereby called from qualified Sri Lankan citizens for recruitment under the merit stream 
to 10 posts in Grade III of the Sri Lanka Administrative Service which have fallen vacant.  
 
 
01. 
In this notification, the term, "Secretary" shall mean the “Secretary to the Ministry in charge of 
the subject of Public Administration”. The term, "Service" shall mean the “Sri Lanka 
Administrative Service”. 
 
02

In [16]:
#Checking other languages cleaning safe
for language in ["English", "Sinhala", "Tamil"]:

    sample = (
        df[df["language"] == language]
        .iloc[0]
    )

    print("\n" + "=" * 70)
    print("Language:", language)

    print("\nORIGINAL:")
    print("-" * 50)
    print(
        sample["text"][:500]
    )

    print("\nCLEANED:")
    print("-" * 50)
    print(
        sample["clean_text"][:500]
    )


Language: English

ORIGINAL:
--------------------------------------------------
Public Administration Circular : 17/2026 
 
 
My number : SLAS/R/Merit/2026 
Ministry of Public Administration, 
Provincial Councils and Local Government  
Independence Square 
Colombo 07. 
 
10.08.2026 
 
Secretaries of Ministries 
Chief Secretaries of Provinces 
Secretaries of Commissions 
District Secretaries / Government Agents 
Heads of Departments. 
 
Competitive Examination for Recruitment to Grade III of the Sri Lanka Administrative Service 
under the Merit Stream -2026 
 
Applications a

CLEANED:
--------------------------------------------------
Public Administration Circular : 17/2026 
My number : SLAS/R/Merit/2026 
Ministry of Public Administration, 
Provincial Councils and Local Government 
Independence Square 
Colombo 07. 
10.08.2026 
Secretaries of Ministries 
Chief Secretaries of Provinces 
Secretaries of Commissions 
District Secretaries / Government Agents 
Heads of Departments. 
Competit

In [17]:
df["original_length"] = (
    df["text"].str.len()
)

df["clean_length"] = (
    df["clean_text"].str.len()
)

df[
    [
        "language",
        "original_length",
        "clean_length"
    ]
].groupby(
    "language"
).mean()

,original_length,clean_length
language,,
English,7236.382088,6425.008376
Sinhala,7311.085052,6228.242268
Tamil,7683.014820,6699.241624


### Sinhala Unicode Preservation Check

In [18]:
sample = (
    df[df["language"] == "Sinhala"]
    .iloc[0]
)

print("ORIGINAL:")
print(sample["text"][:500])

print("\nCLEANED:")
print(sample["clean_text"][:500])

ORIGINAL:
රාජ්‍ය පරිපාලන චක්‍රලේඛ    17/2026 
මලේේ අංකය   SLAS/R/Merit/2026 
රාජ්‍ය පරිපාලන, පළාත් සභා සහ  
පළාත් පාලන අමාත්යාංශය 
නිදහස් චතුරශ්‍රය  
ලේකොළඹ 07 
 
           2026.08.10 
 
 
අමාත්යාංශ ලේඛකම්වරුන්  
පළාත් ප්‍ර ධාන ලේඛකම්වරුන් 
ලේකොමිෂන් සභා ලේඛකම්වරුන් 
දිස්ත්‍රික් ල ලේඛකම්වරුන්   දිසාපතිවරුන් 
ලේදපාර්ත්ලේම්න්තු ප්‍ර ධානීන්. 
 
කුසලත්ා ධාරාව යටලේත් ශ්‍රී ලංකා පරිපාලන ලේසේවලේේ III ලේශ්‍රේණියට බඳවා ගැනීලේම් ත්රඟ 
විභාගය - 2026 
 
ශ්‍රී ලංකා පරිපාලන ලේසේවලේේ III ලේශ්‍රේණිලේේ ත්නතුරුවලට කුසලත්ා

CLEANED:
රාජ්‍ය පරිපාලන චක්‍රලේඛ 17/2026 
මලේේ අංකය SLAS/R/Merit/2026 
රාජ්‍ය පරිපාලන, පළාත් සභා සහ 
පළාත් පාලන අමාත්යාංශය 
නිදහස් චතුරශ්‍රය 
ලේකොළඹ 07 
 2026.08.10 
අමාත්යාංශ ලේඛකම්වරුන් 
පළාත් ප්‍ර ධාන ලේඛකම්වරුන් 
ලේකොමිෂන් සභා ලේඛකම්වරුන් 
දිස්ත්‍රික් ල ලේඛකම්වරුන් දිසාපතිවරුන් 
ලේදපාර්ත්ලේම්න්තු ප්‍ර ධානීන්. 
කුසලත්ා ධාරාව යටලේත් ශ්‍රී ලංකා පරිපාලන ලේසේවලේේ III ලේශ්‍රේණියට බඳවා ගැනීලේම් ත්රඟ 
විභාගය - 2026 
ශ්‍රී ලංකා පරිපාලන ලේසේවලේේ III ලේශ්‍රේණිලේේ ත්නතුරුවලට කුසලත්ා ධාරාව යට

multilingual chunking.

In [ ]:
def create_chunks(
    text,
    max_chars=1000
):

    # Convert line breaks and repeated whitespace
    # into normal spaces
    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()


    # Split text approximately into sentences


    raw_sentences = re.split(
        r"(?<=[.!?])\s+",
        text
    )
    # Join standalone section numbers
    # Example:
    # "02." + "Method of Recruitment:"

    sentences = []

    index = 0


    while index < len(raw_sentences):

        current = (
            raw_sentences[index]
            .strip()
        )


        # Detect standalone section/list numbers:
        # 1.
        # 02.
        # 2.3.
        # 3.1.2.
        if (
            re.fullmatch(
                r"\d+(?:\.\d+)*\.",
                current
            )
            and index + 1 < len(raw_sentences)
        ):

            next_sentence = (
                raw_sentences[index + 1]
                .strip()
            )


            sentences.append(
                current
                + " "
                + next_sentence
            )

            index += 2


        else:

            if current:

                sentences.append(
                    current
                )

            index += 1


    # Build chunks without breaking sentences
    
    chunks = []

    current_chunk = ""


    for sentence in sentences:

        sentence = sentence.strip()


        if not sentence:
            continue

        # Handle unusually long sentences
        # caused by OCR or missing punctuation
        if len(sentence) > max_chars:

            words = sentence.split()

            for word in words:

                if (
                    len(current_chunk)
                    + len(word)
                    + 1
                    <= max_chars
                ):

                    if current_chunk:
                        current_chunk += " "

                    current_chunk += word

                else:

                    if current_chunk:
                        chunks.append(
                            current_chunk.strip()
                        )

                    current_chunk = word

            continue    

        # Add sentence if the chunk
        # remains within the limit
        if (
            len(current_chunk)
            + len(sentence)
            + 1
            <= max_chars
        ):

            if current_chunk:

                current_chunk += " "


            current_chunk += sentence


        else:

            # Save previous chunk
            if current_chunk:

                chunks.append(
                    current_chunk.strip()
                )


            # Start a new chunk
            current_chunk = sentence


    # Save final chunk
    if current_chunk:

        chunks.append(
            current_chunk.strip()
        )


    return chunks

In [27]:
#sample test one english doc
sample_text = (
    df[df["language"] == "English"]
    .iloc[0]["clean_text"]
)

sample_chunks = create_chunks(
    sample_text
)

print(
    "Number of chunks:",
    len(sample_chunks)
)

for index, chunk in enumerate(
    sample_chunks[:3],
    start=1
):

    print(
        f"\nCHUNK {index}"
    )

    print("-" * 60)

    print(chunk)

    print(
        "\nCharacters:",
        len(chunk)
    )

Number of chunks: 25

CHUNK 1
------------------------------------------------------------
Public Administration Circular : 17/2026 My number : SLAS/R/Merit/2026 Ministry of Public Administration, Provincial Councils and Local Government Independence Square Colombo 07. 10.08.2026 Secretaries of Ministries Chief Secretaries of Provinces Secretaries of Commissions District Secretaries / Government Agents Heads of Departments. Competitive Examination for Recruitment to Grade III of the Sri Lanka Administrative Service under the Merit Stream -2026 Applications are hereby called from qualified Sri Lankan citizens for recruitment under the merit stream to 10 posts in Grade III of the Sri Lanka Administrative Service which have fallen vacant. 01. In this notification, the term, "Secretary" shall mean the “Secretary to the Ministry in charge of the subject of Public Administration”. The term, "Service" shall mean the “Sri Lanka Administrative Service”.

Characters: 867

CHUNK 2
---------------

In [28]:
df["chunks"] = (
    df["clean_text"]
    .apply(create_chunks)
)

df["chunk_count"] = (
    df["chunks"]
    .apply(len)
)

In [29]:
print(
    "Total documents:",
    len(df)
)

print(
    "Total chunks:",
    df["chunk_count"].sum()
)

df.groupby(
    "language"
)["chunk_count"].describe()

Total documents: 4656
Total chunks: 34316


,count,mean,std,min,25%,50%,75%,max
language,,,,,,,,
English,1552.0,7.362113,21.416151,1.0,2.0,2.0,5.00,477.0
Sinhala,1552.0,7.137242,21.633721,1.0,2.0,2.0,5.00,456.0
Tamil,1552.0,7.611469,21.723435,1.0,2.0,3.0,5.25,487.0


In [30]:
max_chunk_length = max(
    len(chunk)
    for chunks in df["chunks"]
    for chunk in chunks
)

print(
    "Maximum chunk length:",
    max_chunk_length
)

Maximum chunk length: 1000


In [31]:
# Convert the document-level dataframe
# into a chunk-level dataframe

chunk_df = (
    df[
        [
            "circular_id",
            "circular_number",
            "year",
            "language",
            "pdf_url",
            "extraction_status",
            "chunks"
        ]
    ]
    .explode("chunks")
    .rename(
        columns={
            "chunks": "chunk_text"
        }
    )
    .reset_index(drop=True)
)

print(
    "Chunk-level dataset shape:",
    chunk_df.shape
)

chunk_df.head()

Chunk-level dataset shape: (34316, 7)


,circular_id,circular_number,year,language,pdf_url,extraction_status,chunk_text
0,2221,17/2026,2026,English,https://pubad.gov.lk/web/images/circulars/2026...,DIRECT_TEXT,Public Administration Circular : 17/2026 My nu...
1,2221,17/2026,2026,English,https://pubad.gov.lk/web/images/circulars/2026...,DIRECT_TEXT,02. Method of Recruitment: Recruitment shall b...
2,2221,17/2026,2026,English,https://pubad.gov.lk/web/images/circulars/2026...,DIRECT_TEXT,"2. Establishments Code, Procedural Rules of th..."
3,2221,17/2026,2026,English,https://pubad.gov.lk/web/images/circulars/2026...,DIRECT_TEXT,03. Conditions of Service: 3.1 A selected cand...
4,2221,17/2026,2026,English,https://pubad.gov.lk/web/images/circulars/2026...,DIRECT_TEXT,The candidates should pass the first efficienc...


In [32]:
chunk_df["chunk_index"] = (
    chunk_df
    .groupby(
        [
            "circular_id",
            "language"
        ]
    )
    .cumcount()
)

In [33]:
chunk_df[
    [
        "circular_id",
        "language",
        "chunk_index",
        "chunk_text"
    ]
].head(10)

,circular_id,language,chunk_index,chunk_text
0,2221,English,0,Public Administration Circular : 17/2026 My nu...
1,2221,English,1,02. Method of Recruitment: Recruitment shall b...
2,2221,English,2,"2. Establishments Code, Procedural Rules of th..."
3,2221,English,3,03. Conditions of Service: 3.1 A selected cand...
4,2221,English,4,The candidates should pass the first efficienc...
5,2221,English,5,05. Qualifications for Recruitment: 5.1 Releva...
6,2221,English,6,5.2 Age and Other Qualifications to be Complet...
7,2221,English,7,Officers who have retired from service before ...
8,2221,English,8,It is expected to assess the intelligence leve...
9,2221,English,9,Financial Regulations of the government and Pr...


In [34]:
#Final chunk validation
# Check for empty chunks
empty_chunks = (
    chunk_df["chunk_text"]
    .str.strip()
    .eq("")
    .sum()
)

print(
    "Empty chunks:",
    empty_chunks
)

# Check chunks by language
print(
    "\nChunks by language:"
)

print(
    chunk_df["language"]
    .value_counts()
)

Empty chunks: 0

Chunks by language:
language
Tamil      11813
English    11426
Sinhala    11077
Name: count, dtype: int64


save preprocessing output

In [35]:
output_file = (
    "../data/processed/"
    "multilingual_chunks.jsonl"
)

chunk_df.to_json(
    output_file,
    orient="records",
    lines=True,
    force_ascii=False
)

print(
    "Saved to:",
    output_file
)

print(
    "Total saved chunks:",
    len(chunk_df)
)

Saved to: ../data/processed/multilingual_chunks.jsonl
Total saved chunks: 34316


In [1]:
import pandas as pd

documents_df = pd.read_json(
    "../data/processed/extracted_documents_clean.jsonl",
    lines=True
)

print(documents_df.columns.tolist())

['circular_id', 'circular_number', 'year', 'language', 'pdf_url', 'extraction_status', 'character_count', 'text']


In [2]:
training_doc = documents_df[
    (documents_df["circular_id"] == 2221)
    &
    (documents_df["language"] == "English")
].iloc[0]


print(
    "Stored character_count:",
    training_doc["character_count"]
)

print(
    "Actual text length:",
    len(training_doc["text"])
)

print(
    "\nFirst 500 characters:"
)

print(
    training_doc["text"][:500]
)

Stored character_count: 23362
Actual text length: 23362

First 500 characters:
Public Administration Circular : 17/2026 
 
 
My number : SLAS/R/Merit/2026 
Ministry of Public Administration, 
Provincial Councils and Local Government  
Independence Square 
Colombo 07. 
 
10.08.2026 
 
Secretaries of Ministries 
Chief Secretaries of Provinces 
Secretaries of Commissions 
District Secretaries / Government Agents 
Heads of Departments. 
 
Competitive Examination for Recruitment to Grade III of the Sri Lanka Administrative Service 
under the Merit Stream -2026 
 
Applications a
